# Imports

In [2]:
import os
import pickle
from pathlib import Path

import pandas as pd
import numpy as np

import cv2
import torch

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
from pathlib import Path

datadir = Path("/data/raw")
datadir.list_dir()

# Configuration

In [3]:
class Config:

    # DATA_DIR = Path("/kaggle/input/datasets/organizations/nih-chest-xrays/data")
    DATA_DIR = Path("../data/raw")
    CSV_PATH = DATA_DIR / "Data_Entry_2017.csv"
    ARTIFACT_DIR = Path("../data/artifacts")
    TARGET_LABELS = [
        "Atelectasis",
        "Effusion",
        "Pneumothorax"
    ]
    IMAGE_SIZE = 224
    BATCH_SIZE = 32
    NUM_WORKERS = 4
    RANDOM_STATE = 42

In [4]:
Config.ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Load Data

In [5]:
df = pd.read_csv(Config.CSV_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (112120, 12)


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [9]:
columns_to_keep = [
    "Image Index",
    "Finding Labels",
    "Follow-up #",
    "Patient ID",
    "Patient Age",
    "Patient Gender",
    "View Position"
]

df = df[columns_to_keep]

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112120 entries, 0 to 112119
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   Image Index     112120 non-null  str  
 1   Finding Labels  112120 non-null  str  
 2   Follow-up #     112120 non-null  int64
 3   Patient ID      112120 non-null  int64
 4   Patient Age     112120 non-null  int64
 5   Patient Gender  112120 non-null  str  
 6   View Position   112120 non-null  str  
dtypes: int64(3), str(4)
memory usage: 6.0 MB


In [10]:
def prepare_dataset(
    df: pd.DataFrame,
    target_labels: list,
    patient_col: str = "Patient ID",
    label_col: str = "Finding Labels",
    image_col: str = "Image Index",
    drop_columns: list = None,
    val_size: float = 0.15,
    test_size: float = 0.15,
    random_state: int = 42
):

    df = df.copy()

    if drop_columns:
        df = df.drop(columns=drop_columns, errors="ignore")

    relevant_mask = df[label_col].apply(
        lambda x: any(label in x.split("|") for label in target_labels)
        or "No Finding" in x.split("|")
    )

    df = df[relevant_mask].reset_index(drop=True)

    def encode_labels(label_str):
        labels = label_str.split("|")
        return [1 if l in labels else 0 for l in target_labels]

    labels_df = df[label_col].apply(encode_labels)
    labels_df = pd.DataFrame(labels_df.tolist(), columns=target_labels, index=df.index)

    df = pd.concat([df, labels_df], axis=1)

    patients = df[patient_col].unique()

    train_patients, temp_patients = train_test_split(
        patients,
        test_size=val_size + test_size,
        random_state=random_state
    )

    val_patients, test_patients = train_test_split(
        temp_patients,
        test_size=test_size / (val_size + test_size),
        random_state=random_state
    )

    df["split"] = df[patient_col].map(
        lambda pid:
        "train" if pid in train_patients
        else "val" if pid in val_patients
        else "test"
    )

    return df

In [11]:
prepared_df = prepare_dataset(
    df,
    Config.TARGET_LABELS,
    random_state=Config.RANDOM_STATE
)

prepared_df.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,Atelectasis,Effusion,Pneumothorax,split
0,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,0,1,0,train
1,00000002_000.png,No Finding,0,2,81,M,PA,0,0,0,train
2,00000005_000.png,No Finding,0,5,69,F,PA,0,0,0,train
3,00000005_001.png,No Finding,1,5,69,F,AP,0,0,0,train
4,00000005_002.png,No Finding,2,5,69,F,AP,0,0,0,train


In [12]:
train_df = prepared_df[prepared_df["split"] == "train"]
val_df = prepared_df[prepared_df["split"] == "val"]
test_df = prepared_df[prepared_df["split"] == "test"]

print(len(train_df), len(val_df), len(test_df))

59975 13073 12675


In [13]:
train_patients = set(train_df["Patient ID"])
val_patients = set(val_df["Patient ID"])
test_patients = set(test_df["Patient ID"])

print("Train-Val overlap:", len(train_patients & val_patients))
print("Train-Test overlap:", len(train_patients & test_patients))
print("Val-Test overlap:", len(val_patients & test_patients))

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


In [14]:
for label in Config.TARGET_LABELS:

    print(label)

    print("Train:", train_df[label].mean())
    print("Val:", val_df[label].mean())
    print("Test:", test_df[label].mean())

    print()

Atelectasis
Train: 0.13443934972905378
Val: 0.12958005048573396
Test: 0.14216962524654833

Effusion
Train: 0.15774906210921216
Val: 0.13998317142201483
Test: 0.1598422090729783

Pneumothorax
Train: 0.0596748645268862
Val: 0.07549912032433259
Test: 0.058067061143984224



In [15]:
def build_image_index(cache_file):

    if Path(cache_file).exists():

        print("Loading cached image index...")

        with open(cache_file, "rb") as f:
            return pickle.load(f)

    print("Building image index...")

    image_index = {}

    for folder in os.listdir(Config.DATA_DIR):

        if folder.startswith("images_"):

            images_path = Config.DATA_DIR / folder / "images"

            for file in os.listdir(images_path):

                image_index[file] = str(images_path / file)

    with open(cache_file, "wb") as f:
        pickle.dump(image_index, f)

    return image_index

In [16]:
image_index = build_image_index(
    Config.ARTIFACT_DIR / "image_index.pkl"
)

print("Indexed images:", len(image_index))

Building image index...


Indexed images: 112120


In [21]:
img_idx = pd.read_pickle(Config.ARTIFACT_DIR / "image_index.pkl")
img_idx

{'00000001_000.png': '..\\data\\raw\\images_001\\images\\00000001_000.png',
 '00000001_001.png': '..\\data\\raw\\images_001\\images\\00000001_001.png',
 '00000001_002.png': '..\\data\\raw\\images_001\\images\\00000001_002.png',
 '00000002_000.png': '..\\data\\raw\\images_001\\images\\00000002_000.png',
 '00000003_000.png': '..\\data\\raw\\images_001\\images\\00000003_000.png',
 '00000003_001.png': '..\\data\\raw\\images_001\\images\\00000003_001.png',
 '00000003_002.png': '..\\data\\raw\\images_001\\images\\00000003_002.png',
 '00000003_003.png': '..\\data\\raw\\images_001\\images\\00000003_003.png',
 '00000003_004.png': '..\\data\\raw\\images_001\\images\\00000003_004.png',
 '00000003_005.png': '..\\data\\raw\\images_001\\images\\00000003_005.png',
 '00000003_006.png': '..\\data\\raw\\images_001\\images\\00000003_006.png',
 '00000003_007.png': '..\\data\\raw\\images_001\\images\\00000003_007.png',
 '00000004_000.png': '..\\data\\raw\\images_001\\images\\00000004_000.png',
 '00000005_0

In [15]:
missing = [
    img for img in prepared_df["Image Index"]
    if img not in image_index
]

print("Missing images:", len(missing))

Missing images: 0


In [16]:
class ChestXrayDataset(Dataset):

    def __init__(self, df, image_index, target_labels, transform=None):

        self.df = df.reset_index(drop=True)

        self.image_index = image_index

        self.target_labels = target_labels

        self.transform = transform

        self.image_names = self.df["Image Index"].values

        self.labels = self.df[self.target_labels].values.astype("float32")

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        image_name = self.image_names[idx]

        image_path = self.image_index[image_name]

        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise RuntimeError(f"Failed to load {image_path}")

        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

        if self.transform:
            image = self.transform(image)

        label = torch.from_numpy(self.labels[idx])

        return image, label

In [17]:
def get_train_transforms(image_size):

    return transforms.Compose([

        transforms.ToPILImage(),

        transforms.RandomResizedCrop(image_size, scale=(0.9,1.0)),

        transforms.RandomHorizontalFlip(p=0.3),

        transforms.RandomRotation(7),

        transforms.RandomAffine(
            degrees=5,
            translate=(0.02,0.02),
            scale=(0.98,1.02)
        ),

        transforms.ColorJitter(
            brightness=0.05,
            contrast=0.05
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[0.485,0.456,0.406],
            std=[0.229,0.224,0.225]
        )
    ])

In [18]:
def get_val_transforms(image_size):

    return transforms.Compose([

        transforms.ToPILImage(),

        transforms.Resize(int(image_size * 1.14)),

        transforms.CenterCrop(image_size),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[0.485,0.456,0.406],
            std=[0.229,0.224,0.225]
        )
    ])

In [19]:
def create_dataloaders(
    df,
    image_index,
    target_labels,
    batch_size,
    image_size,
    num_workers=4
):

    train_dataset = ChestXrayDataset(
        df[df["split"] == "train"],
        image_index,
        target_labels,
        transform=get_train_transforms(image_size)
    )

    val_dataset = ChestXrayDataset(
        df[df["split"] == "val"],
        image_index,
        target_labels,
        transform=get_val_transforms(image_size)
    )

    test_dataset = ChestXrayDataset(
        df[df["split"] == "test"],
        image_index,
        target_labels,
        transform=get_val_transforms(image_size)
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    return train_loader, val_loader, test_loader

In [20]:
train_loader, val_loader, test_loader = create_dataloaders(
    prepared_df,
    image_index,
    Config.TARGET_LABELS,
    Config.BATCH_SIZE,
    Config.IMAGE_SIZE,
    Config.NUM_WORKERS
)

In [21]:
images, labels = next(iter(train_loader))

print("Image batch:", images.shape)
print("Label batch:", labels.shape)

Image batch: torch.Size([32, 3, 224, 224])
Label batch: torch.Size([32, 3])


In [22]:
with open(Config.ARTIFACT_DIR / "prepared_df.pkl", "wb") as f:
    pickle.dump(prepared_df, f)

with open(Config.ARTIFACT_DIR / "train_df.pkl", "wb") as f:
    pickle.dump(train_df, f)

with open(Config.ARTIFACT_DIR / "val_df.pkl", "wb") as f:
    pickle.dump(val_df, f)

with open(Config.ARTIFACT_DIR / "test_df.pkl", "wb") as f:
    pickle.dump(test_df, f)

In [23]:
print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

print("Target labels:", Config.TARGET_LABELS)

Training samples: 59975
Validation samples: 13073
Test samples: 12675
Target labels: ['Atelectasis', 'Effusion', 'Pneumothorax']
